In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/cifar-10/trainLabels.csv
/kaggle/input/competitions/cifar-10/sampleSubmission.csv
/kaggle/input/competitions/cifar-10/test.7z
/kaggle/input/competitions/cifar-10/train.7z


In [10]:
import os

In [11]:
!apt-get install p7zip-full -y
!7z x /kaggle/input/competitions/cifar-10/train.7z -o./

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,4 CPUs Intel(R) Xeon(R) CPU @ 2.00GHz (50653),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /kaggle/input/competitions/cifar-1                                            1 file, 109723070 bytes (105 MiB)

Extracting archive: /kaggle/input/competitions/cifar-10/train.7z
--
Path = /kaggle/input/competitions/cifar-10/train.7z
Type = 7z
Physical Size = 109723070
Headers Size = 294768
Method = LZMA:26
Solid = +
Blocks = 1

          2% 755 - train/10678.p                          2% 1280 - train/1115.p                          2% 1648 - train/11481.pn                            4% 1868 - train/1168.p                          6% 1

In [12]:
import torch
import pandas as pd
import os
import glob
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [14]:
class CIFAR10Kaggle(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        self.label_mapping = {
            'airplane':0, 'automobile':1, 'bird':2, 'cat':3, 'deer':4,
            'dog':5, 'frog':6, 'horse':7, 'ship':8, 'truck':9
        }

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = str(self.df.iloc[idx, 0])
        img_name = os.path.join(self.img_dir, img_id + ".png")
        image = Image.open(img_name)
        label = self.label_mapping[self.df.iloc[idx, 1]]
        if self.transform:
            image = self.transform(image)
        return image, label

In [15]:
train_csv = '/kaggle/input/competitions/cifar-10/trainLabels.csv'
train_dir = './train' 

In [16]:
from torchvision import models

In [17]:
# 1. Define the device (This fixes your NameError)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Load the pre-trained ResNet18 model
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 3. Modify the final layer
# ResNet-18's last layer is named 'fc' (Fully Connected)
num_features = model.fc.in_features 
model.fc = nn.Linear(num_features, 10) 

# 4. Move the model to the device
model = model.to(device)

Using device: cuda


In [18]:
# Updated Transform for ResNet
transform = transforms.Compose([
    transforms.Resize((64, 64)), # Upscaling helps pre-trained models
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # ImageNet Stats
])

# Re-initialize datasets with the new transform
full_dataset = CIFAR10Kaggle(csv_file=train_csv, img_dir=train_dir, transform=transform)
train_data, val_data = random_split(full_dataset, [45000, 5000])
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)

# Optimizer (Using a smaller learning rate for fine-tuning)
optimizer = optim.Adam(model.parameters(), lr=0.0001) 
criterion = nn.CrossEntropyLoss()

# Training Loop
for epoch in range(5): # Pre-trained models converge very fast
    model.train()
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
    # --- VALIDATION ---
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    print(f"Epoch {epoch+1} | Val Accuracy: {100 * correct / total:.2f}%")

Epoch 1 | Val Accuracy: 88.34%
Epoch 2 | Val Accuracy: 89.58%
Epoch 3 | Val Accuracy: 89.16%
Epoch 4 | Val Accuracy: 89.72%
Epoch 5 | Val Accuracy: 89.46%


In [20]:
# Save the weights only (Recommended)
torch.save(model.state_dict(), 'cifar10_resnet18.pth')
print("Model saved to cifar10_resnet18.pth")

Model saved to cifar10_resnet18.pth
